# Bible Data

This notebook constructs an English-Igbo Bible parallel dataset by aligning verses from two VPL sources.

The final output keeps only `English` and `Igbo` fields so that the resulting file matches the structure of the Jehovah's Witness dataset used elsewhere in the project. Alignment is performed verse by verse using `verseID`.

## 1. Package setup

In [21]:
# Run once if these libraries are not already installed.
# %pip install -q pandas

## 2. Imports and file paths

In [22]:
from pathlib import Path
import re
import unicodedata
import zipfile

import pandas as pd
from IPython.display import display

ENGLISH_SQL_PATH = Path(r"C:\Users\Mr. Paul\Downloads\engwebp_vpl\engwebp_vpl.sql")
IGBO_SQL_PATH = Path(r"C:\Users\Mr. Paul\Downloads\ibo_vpl\ibo_vpl.sql")
IGBO_ZIP_PATH = Path(r"C:\Users\Mr. Paul\Downloads\ibo_vpl.zip")

TRAIN_OUTPUT_PATH = Path(r"C:\Users\Mr. Paul\Downloads\Bible_train_cleaned.jsonl")
TEST_OUTPUT_PATH = Path(r"C:\Users\Mr. Paul\Downloads\Bible_test_cleaned.jsonl")

TEST_FRACTION = 0.10
RANDOM_SEED = 42

## 3. Verify source availability

In [23]:
print(f"English SQL exists: {ENGLISH_SQL_PATH.exists()} -> {ENGLISH_SQL_PATH}")
print(f"Igbo SQL exists:    {IGBO_SQL_PATH.exists()} -> {IGBO_SQL_PATH}")
print(f"Igbo ZIP exists:    {IGBO_ZIP_PATH.exists()} -> {IGBO_ZIP_PATH}")

if not ENGLISH_SQL_PATH.exists():
    raise FileNotFoundError("The English Bible SQL file was not found.")

if not IGBO_SQL_PATH.exists() and not IGBO_ZIP_PATH.exists():
    raise FileNotFoundError("Neither the Igbo SQL file nor the Igbo ZIP file was found.")

English SQL exists: True -> C:\Users\Mr. Paul\Downloads\engwebp_vpl\engwebp_vpl.sql
Igbo SQL exists:    False -> C:\Users\Mr. Paul\Downloads\ibo_vpl\ibo_vpl.sql
Igbo ZIP exists:    True -> C:\Users\Mr. Paul\Downloads\ibo_vpl.zip


## 4. Helper functions

The SQL files store one verse per `INSERT` statement. The helper functions below parse those lines, normalize punctuation and whitespace, and return a verse-level table.

In [24]:
VERSE_PATTERN = re.compile(
    r'INSERT INTO \w+ VALUES \("([^"]*)","([^"]*)","([^"]*)","([^"]*)","([^"]*)","([^"]*)","(.*)"\);$'
)


def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_vpl_sql(lines, source_name):
    rows = []

    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()

        if not line.startswith("INSERT INTO"):
            continue

        match = VERSE_PATTERN.match(line)
        if not match:
            raise ValueError(
                f"Could not parse line {line_number} from {source_name}: {line[:120]}"
            )

        verse_id, canon_order, book, chapter, start_verse, end_verse, verse_text = match.groups()

        rows.append(
            {
                "verseID": verse_id,
                "canon_order": canon_order,
                "book": book,
                "chapter": int(chapter),
                "startVerse": int(start_verse),
                "endVerse": int(end_verse),
                "verseText": normalize_text(verse_text),
            }
        )

    return pd.DataFrame(rows)


def load_english_verses():
    with ENGLISH_SQL_PATH.open(encoding="utf-8") as handle:
        return parse_vpl_sql(handle, ENGLISH_SQL_PATH.name)


def load_igbo_verses():
    if IGBO_SQL_PATH.exists():
        with IGBO_SQL_PATH.open(encoding="utf-8") as handle:
            return parse_vpl_sql(handle, IGBO_SQL_PATH.name)

    with zipfile.ZipFile(IGBO_ZIP_PATH) as archive:
        with archive.open("ibo_vpl.sql") as handle:
            lines = (line.decode("utf-8") for line in handle)
            return parse_vpl_sql(lines, f"{IGBO_ZIP_PATH.name}::ibo_vpl.sql")

## 5. Load the English and Igbo verse tables

In [25]:
english_df = load_english_verses().rename(columns={"verseText": "English"})
igbo_df = load_igbo_verses().rename(columns={"verseText": "Igbo"})

pd.DataFrame(
    {
        "source": ["English Bible", "Igbo Bible"],
        "rows": [len(english_df), len(igbo_df)],
        "columns": [list(english_df.columns), list(igbo_df.columns)],
    }
)

,source,rows,columns
0,English Bible,31098,"[verseID, canon_order, book, chapter, startVer..."
1,Igbo Bible,31103,"[verseID, canon_order, book, chapter, startVer..."


## 6. Check verse alignment

The two editions are close, but not perfectly identical. This step measures verse coverage before pairing the data.

In [26]:
english_ids = set(english_df["verseID"])
igbo_ids = set(igbo_df["verseID"])

missing_in_igbo = sorted(english_ids - igbo_ids)
missing_in_english = sorted(igbo_ids - english_ids)

alignment_summary = pd.DataFrame(
    {
        "metric": [
            "English verses",
            "Igbo verses",
            "Verse IDs shared by both corpora",
            "Verse IDs missing in Igbo",
            "Verse IDs missing in English",
        ],
        "value": [
            len(english_ids),
            len(igbo_ids),
            len(english_ids & igbo_ids),
            len(missing_in_igbo),
            len(missing_in_english),
        ],
    }
)

display(alignment_summary)

display(
    pd.DataFrame(
        {
            "missing_in_igbo": pd.Series(missing_in_igbo[:8]),
            "missing_in_english": pd.Series(missing_in_english[:8]),
        }
    )
)

,metric,value
0,English verses,31098
1,Igbo verses,31103
2,Verse IDs shared by both corpora,31095
3,Verse IDs missing in Igbo,3
4,Verse IDs missing in English,8


,missing_in_igbo,missing_in_english
0,RM14_24,AC15_34
1,RM14_25,AC24_7
2,RM14_26,AC8_37
3,NaN,J31_15
4,NaN,LK17_36
5,NaN,RM16_25
6,NaN,RM16_26
7,NaN,RM16_27


## 7. Build verse-aligned English-Igbo pairs

An inner join on `verseID` keeps only verses that appear in both corpora. This is the safest approach when the two editions do not have identical verse numbering.

In [27]:
aligned_df = english_df.merge(
    igbo_df[["verseID", "Igbo"]],
    on="verseID",
    how="inner",
)

aligned_df = aligned_df[
    ["verseID", "canon_order", "book", "chapter", "startVerse", "endVerse", "English", "Igbo"]
]

aligned_df.head()

,verseID,canon_order,book,chapter,startVerse,endVerse,English,Igbo
0,GN1_1,002_1_1,GEN,1,1,1,"In the beginning, God created the heavens and ...",Na mmalite Chineke kere eluigwe na ụwa.
1,GN1_2,002_1_2,GEN,1,2,2,The earth was formless and empty. Darkness was...,"Ụwa bụ ihe na-enweghị ụdịdị ọbụla, bụrụkwa ihe..."
2,GN1_3,002_1_3,GEN,1,3,3,"God said, ""Let there be light,"" and there was ...","Mgbe ahụ Chineke kwuru sị, ""Ka ìhè dị,"" ìhè dị..."
3,GN1_4,002_1_4,GEN,1,4,4,"God saw the light, and saw that it was good. G...",Chineke hụrụ na ìhè ahụ mara mma. Ọ kpara oke ...
4,GN1_5,002_1_5,GEN,1,5,5,"God called the light ""day"", and the darkness h...","Chineke kpọrọ ìhè ahụ Ehihie, kpọọ ọchịchịrị a..."


## 8. Clean the paired verses

For this project, the main cleaning step is to remove rows where either side is empty after normalization. Verses with multiple sentences are kept intact.

In [28]:
clean_bible_df = aligned_df[
    aligned_df["English"].str.strip().ne("") & aligned_df["Igbo"].str.strip().ne("")
].copy()

cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "Aligned verse pairs",
            "Pairs after removing blanks",
            "Duplicate English-Igbo text pairs",
        ],
        "value": [
            len(aligned_df),
            len(clean_bible_df),
            int(clean_bible_df[["English", "Igbo"]].duplicated().sum()),
        ],
    }
)

display(cleaning_summary)

,metric,value
0,Aligned verse pairs,31095
1,Pairs after removing blanks,31095
2,Duplicate English-Igbo text pairs,168


## 9. Create cleaned Bible pairs and split the data

The final dataset keeps only the two translation columns so that the structure matches the JHW data. A reproducible random split is then used to create train and test files.

In [29]:
bible_pairs_df = clean_bible_df[["English", "Igbo"]].copy().reset_index(drop=True)

shuffled_bible_pairs_df = bible_pairs_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_size = int(len(shuffled_bible_pairs_df) * TEST_FRACTION)

Bible_test_cleaned = shuffled_bible_pairs_df.iloc[:test_size].copy().reset_index(drop=True)
Bible_train_cleaned = shuffled_bible_pairs_df.iloc[test_size:].copy().reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "split": ["Bible_train_cleaned", "Bible_test_cleaned"],
        "rows": [len(Bible_train_cleaned), len(Bible_test_cleaned)],
    }
)

display(split_summary)
display(Bible_train_cleaned.head())
display(Bible_test_cleaned.head())

,English,Igbo
0,"In the beginning, God created the heavens and ...",Na mmalite Chineke kere eluigwe na ụwa.
1,The earth was formless and empty. Darkness was...,"Ụwa bụ ihe na-enweghị ụdịdị ọbụla, bụrụkwa ihe..."
2,"God said, ""Let there be light,"" and there was ...","Mgbe ahụ Chineke kwuru sị, ""Ka ìhè dị,"" ìhè dị..."
3,"God saw the light, and saw that it was good. G...",Chineke hụrụ na ìhè ahụ mara mma. Ọ kpara oke ...
4,"God called the light ""day"", and the darkness h...","Chineke kpọrọ ìhè ahụ Ehihie, kpọọ ọchịchịrị a..."


,English,Igbo
14600,"Strap your sword on your thigh, O mighty one, ...","Kenye mma agha gị n'ukwu gị, gị onye dị ike; y..."
24523,"He took hold of the blind man by the hand, and...",Ọ duuru onye ìsì ahụ n'aka pụọ n'azụ obodo. Mg...
3058,The priest shall examine him again on the seve...,"Mgbe abalị asaa nke a gafekwara, onye nchụaja ..."
4810,The LORD spoke to Moses in the plains of Moab ...,"N'obosara ala Moab, n'akụkụ Jọdan, na ncherita..."
7682,"As they came, when David returned from the sla...","Mgbe ndị agha ahụ na-alọghachite, mgbe Devid g..."


## 10. Save the cleaned train and test files

In [30]:
Bible_train_cleaned.to_json(TRAIN_OUTPUT_PATH, orient="records", lines=True, force_ascii=False)
Bible_test_cleaned.to_json(TEST_OUTPUT_PATH, orient="records", lines=True, force_ascii=False)

print(f"Saved train split to: {TRAIN_OUTPUT_PATH}")
print(f"Saved test split to:  {TEST_OUTPUT_PATH}")
print(f"Train rows: {len(Bible_train_cleaned):,}")
print(f"Test rows: {len(Bible_test_cleaned):,}")

Saved Bible English-Igbo pairs to: C:\Users\Mr. Paul\Downloads\bible_english_to_igbo_pairs.jsonl
Total saved rows: 31,095
